# 02 - Baseline Model & Class Imbalance Experiments

## Goals
1. Build baseline Logistic Regression model
2. Demonstrate why accuracy is a misleading metric for imbalanced data
3. Compare class imbalance handling strategies:
   - No handling (baseline)
   - Random undersampling
   - SMOTE oversampling
   - Class weighting
   - **Cost-sensitive learning** (chosen approach)
4. Establish evaluation protocol (Precision-Recall AUC)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

mlflow.set_experiment('fraud-detection-baseline')
DATA_PATH = Path('../data/raw/creditcard.csv')

## 1. Data Preparation

In [ ]:
df = pd.read_csv(DATA_PATH)

# Scale Amount and Time
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])
df = df.drop(['Amount', 'Time'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']

# Temporal split (important: do NOT random split - use time ordering)
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train size: {len(X_train):,} | Fraud rate: {y_train.mean():.4f}')
print(f'Test size: {len(X_test):,}  | Fraud rate: {y_test.mean():.4f}')

## 2. Why Accuracy Fails (The Imbalance Trap)

In [ ]:
# TODO: Show that predicting ALL legitimate achieves 99.8% accuracy
# TODO: Show precision-recall trade-off
# TODO: Motivate using PR-AUC instead of ROC-AUC

## 3. Baseline: No Imbalance Handling

In [ ]:
with mlflow.start_run(run_name='baseline-no-handling'):
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    
    mlflow.log_metrics({'roc_auc': roc_auc, 'pr_auc': pr_auc})
    print(f'ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}')

## 4. Strategy Comparison

In [ ]:
# TODO: Random Undersampling experiment
# TODO: SMOTE experiment
# TODO: Class weights experiment
# TODO: Cost-sensitive objective experiment
# TODO: Results comparison table

## 5. Results Summary

| Strategy | ROC-AUC | PR-AUC | Recall@0.5 | Notes |
|----------|---------|--------|------------|-------|
| No handling | TBD | TBD | TBD | Baseline |
| Undersampling | TBD | TBD | TBD | Loses data |
| SMOTE | TBD | TBD | TBD | Synthetic data risk |
| Class weights | TBD | TBD | TBD | Good, simple |
| **Cost-sensitive** | TBD | TBD | TBD | **Best: business-aligned** |

→ `03_optimization.ipynb`: XGBoost + hyperparameter tuning + threshold optimization